# Segmented Search Model — Full Equilibrium Solution (Two-way Search)

This notebook solves the two-sector (skilled / unskilled) search model at the
calibrated parameter values and visualises the stationary equilibrium.

All model code is loaded from the `solver/` directory — this notebook
only sets parameters, calls the solver, and plots results.

## 1 · Packages

In [1]:
using LinearAlgebra, SparseArrays, Statistics, Random
using Distributions, FastGaussQuadrature, Interpolations
using Parameters, Printf
using Plots, LaTeXStrings
using Base.Threads
using CSV, DataFrames

Random.seed!(1234)

TaskLocalRNG()

## 2 · Load solver modules

In [2]:
# Notebook lives in  notebooks/
# Solver  lives in  solver/
const NOTEBOOKS_DIR = @__DIR__
const SOLVER_DIR = joinpath(NOTEBOOKS_DIR, "..", "solver")
const SMM_DIR = joinpath(NOTEBOOKS_DIR, "..", "smm")
const PROJECT_ROOT = joinpath(NOTEBOOKS_DIR, "..", "..")
const OUTPUT_DIR   = joinpath(PROJECT_ROOT, "output")
const ESTIMATES_DIR = joinpath(OUTPUT_DIR, "tables")
const DERIVED_DIR = joinpath(PROJECT_ROOT, "data", "derived")

include(joinpath(SOLVER_DIR, "grids.jl"))
include(joinpath(SOLVER_DIR, "params.jl"))
include(joinpath(SOLVER_DIR, "unskilled.jl"))
include(joinpath(SOLVER_DIR, "skilled.jl"))
include(joinpath(SOLVER_DIR, "solver.jl"))
include(joinpath(SOLVER_DIR, "equilibrium.jl"))
include(joinpath(SOLVER_DIR, "single_run_plots.jl"))
include(joinpath(SMM_DIR, "moments.jl"))
include(joinpath(SMM_DIR, "smm_params.jl"))  # ← must come before smm.jl
include(joinpath(SMM_DIR, "smm.jl"))

println("Solver loaded  |  threads: ", Threads.nthreads())

Solver loaded  |  threads: 10


## 3 · Parameters → Solve → Moment Diagnostics

Edit `sim`, `DEFAULT_PARAMS`, `ACTIVE_SCENARIO`, and `ACTIVE_WEIGHTS` at the top of this cell, then run it once — the solve, equilibrium objects, moment table, and Q all update atomically.

In [3]:
# ── Solver settings (edit tolerances here if needed) ────────────────────────
sim = SimParams(
    tol_inner      = 1e-7,
    tol_outer_U    = 1e-6,
    tol_outer_S    = 1e-6,
    tol_global     = 1e-4,
    maxit_inner    = 500,
    maxit_outer    = 300,
    maxit_global   = 50,
    conv_streak    = 1,
    use_anderson   = true,
    anderson_m     = 1,
    anderson_reg   = 1e-10,
    damp_pstar_U   = 1.00,
    damp_pstar_S   = 0.50,
    verbose        = 2,
    verbose_stride = 10,
)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  3  CALIBRATED PARAMETERS                                                   ║
# ║  Change ACTIVE_SCENARIO and/or ACTIVE_WEIGHTS, then re-run this cell.       ║
# ║                                                                              ║
# ║  ACTIVE_SCENARIO : :none | :base_fc | :crisis_fc | :base_covid | :crisis_covid ║
# ║  ACTIVE_WEIGHTS  : :full | :diagonal | :compressed | :equal                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Switches (plain variables — safe to re-run after changing) ────────────────
# Keep SKIP_MOMENTS in sync with smm_main.jl so W and the loss function agree.
SKIP_MOMENTS = Symbol[
    #:ur_total,
    :exp_ur_total,
    :exp_ur_U,
    #:ur_U,
    :exp_ur_S,
    #:ur_S,
    #:exp_ur_S,
    #:skilled_share,
    #:training_share,
    #:emp_var_U,
    #:emp_var_S,
    #:emp_cm3_U,
    #:emp_cm3_S,
    #:ee_rate_S,
    #:p25_wage_U,
    #:p25_wage_S,
    #:p50_wage_U,
    #:p50_wage_S,
    #:mean_wage_U,
    #:mean_wage_S,
    #:sep_rate_S,
    #:jfr_S,
    #:jfr_U,
    #:sep_rate_U,
    #:wage_premium,
    #:theta_U,
    #:theta_S,
]

ACTIVE_SCENARIO = :crisis_fc  # ← edit here
ACTIVE_WEIGHTS  = :equal        # ← edit here
USE_DEFAULT_PARAMS = false   # ← use the ones entered here instead of loading from CSV (for testing) 

# ── Static lookup tables (const — identical every run, no redefinition error) ─
const DEFAULT_PARAMS = Dict{Symbol,Float64}(
    :r         => 0.00417,
    :nu        => 0.03841,
    :phi       => 0.02222,
    :a_l       => 1.46357,
    :b_l       => 4.90168,
    :c         => 3.50000,
    :PU        => 1.01122,
    :gamma_PS  => 2.58787,
    :bU        => 0.00000,
    :bT        => 0.70000,
    :bS        => 1.00000,
    :alpha_U   => 6.32697,
    :a_Gam     => 2.81289,
    :b_Gam     => 0.97088,
    :unsk_mu   => 0.70375,
    :unsk_eta  => 0.50000,
    :unsk_k    => 0.06274,
    :unsk_bet  => 0.50000,
    :unsk_lam  => 0.02563,
    :skl_mu    => 1.45977,
    :skl_eta   => 0.50000,
    :skl_k     => 0.15255,
    :skl_bet   => 0.50000,
    :skl_xi    => 0.00000,
    :skl_lam   => 0.09548,
    :skl_sig   => 1.10000,
)

const PARAM_SPECS = [
    (:r,        "discount rate r"),
    (:nu,       "demographic exit ν"),
    (:phi,      "training completion φ"),
    (:a_l,      "worker type shape a_ℓ"),
    (:b_l,      "worker type shape b_ℓ"),
    (:c,        "training cost c"),
    (:PU,       "unskilled productivity PU"),
    (:gamma_PS, "skilled productivity γ_PS"),
    (:bU,       "unskilled UI flow bU"),
    (:bT,       "training flow bT"),
    (:bS,       "skilled UI flow bS"),
    (:alpha_U,  "damage shock shape α_U"),
    (:a_Gam,    "skilled offer shape a_Γ"),
    (:b_Gam,    "skilled offer shape b_Γ"),
    (:unsk_mu,  "unskilled matching eff μ_U"),
    (:unsk_eta, "unskilled matching elas η_U"),
    (:unsk_k,   "unskilled vacancy cost k_U"),
    (:unsk_bet, "unskilled bargaining β_U"),
    (:unsk_lam, "unskilled damage rate λ_U"),
    (:skl_mu,   "skilled matching eff μ_S"),
    (:skl_eta,  "skilled matching elas η_S"),
    (:skl_k,    "skilled vacancy cost k_S"),
    (:skl_bet,  "skilled bargaining β_S"),
    (:skl_xi,   "skilled exog sep rate ξ_S"),
    (:skl_lam,  "skilled quality shock λ_S"),
    (:skl_sig,  "OJS flow cost σ"),
]

const PARAM_KEYS   = first.(PARAM_SPECS)
const PARAM_LABELS = last.(PARAM_SPECS)

const SCENARIO_TAGS = Dict{Symbol,String}(
    :base_fc      => "base_fc",
    :crisis_fc    => "crisis_fc",
    :base_covid   => "base_covid",
    :crisis_covid => "crisis_covid",
)

const W_SUFFIX = Dict{Symbol,String}(
    :full       => "_fullW",
    :diagonal   => "_diagonalW",
    :compressed => "_compressedW",
    :equal      => "_equalW",
)

# Maps (block, csv-name) → DEFAULT_PARAMS key
const CSV_KEY_MAP = Dict{Tuple{String,String}, Symbol}(
    # CommonParams
    ("common","a_ℓ") => :a_l,     # unicode name as written by save_results
    ("common","b_ℓ") => :b_l,     # unicode name as written by save_results
    ("common","a")   => :a_l,     # ASCII fallback
    ("common","b")   => :b_l,     # ASCII fallback
    ("common","c")   => :c,
    # RegimeParams
    ("regime","PU")      => :PU,
    ("regime","gamma_PS") => :gamma_PS,
    ("regime","bU")  => :bU,
    ("regime","bT")  => :bT,
    ("regime","bS")  => :bS,
    ("regime","U")   => :alpha_U,
    ("regime","α_U") => :alpha_U,
    ("regime","a_Γ") => :a_Gam,   # unicode name as written by save_results
    ("regime","b_Γ") => :b_Gam,   # unicode name as written by save_results
    ("regime","a")   => :a_Gam,   # ASCII fallback
    ("regime","b")   => :b_Gam,   # ASCII fallback
    # UnskilledParams
    ("unsk","μ")  => :unsk_mu,   ("unsk","mu")  => :unsk_mu,
    ("unsk","η")  => :unsk_eta,  ("unsk","eta") => :unsk_eta,
    ("unsk","k")  => :unsk_k,
    ("unsk","β")  => :unsk_bet,  ("unsk","bet") => :unsk_bet,
    ("unsk","λ")  => :unsk_lam,  ("unsk","lam") => :unsk_lam,
    # SkilledParams
    ("skl","μ")  => :skl_mu,    ("skl","mu")  => :skl_mu,
    ("skl","η")  => :skl_eta,   ("skl","eta") => :skl_eta,
    ("skl","k")  => :skl_k,
    ("skl","β")  => :skl_bet,   ("skl","bet") => :skl_bet,
    ("skl","ξ")  => :skl_xi,    ("skl","xi")  => :skl_xi,
    ("skl","λ")  => :skl_lam,   ("skl","lam") => :skl_lam,
    ("skl","σ")  => :skl_sig,   ("skl","sig") => :skl_sig,
    # Fixed-block entries for standard calibrated params
    ("fixed","r") => :r,
    ("fixed","ν") => :nu,   ("fixed","nu")  => :nu,
    ("fixed","φ") => :phi,  ("fixed","phi") => :phi,
    # Fixed-block entries for deep structural params pinned during crisis estimation.
    # save_results writes these with block="fixed" and the unicode field name as key.
    ("fixed","a_ℓ")   => :a_l,
    ("fixed","b_ℓ")   => :b_l,
    ("fixed","c")     => :c,
    ("fixed","bU")    => :bU,
    ("fixed","bT")    => :bT,
    ("fixed","bS")    => :bS,
    ("fixed","unsk_μ") => :unsk_mu,
    ("fixed","unsk_η") => :unsk_eta,
    ("fixed","unsk_β") => :unsk_bet,
    ("fixed","skl_μ")  => :skl_mu,
    ("fixed","skl_η")  => :skl_eta,
    ("fixed","skl_β")  => :skl_bet,
    ("fixed","skl_σ")  => :skl_sig,
)

# ── Loader ────────────────────────────────────────────────────────────────────
function load_scenario_params(scenario::Symbol, weights::Symbol)
    scenario === :none && return copy(DEFAULT_PARAMS)

    haskey(SCENARIO_TAGS, scenario) ||
        error("Unknown scenario :$scenario. Valid: :none, " *
              join((":$s" for s in keys(SCENARIO_TAGS)), ", "))
    haskey(W_SUFFIX, weights) ||
        error("Unknown weights :$weights. Valid: " *
              join((":$s" for s in keys(W_SUFFIX)), ", "))

    fname = "smm_estimates_$(SCENARIO_TAGS[scenario])$(W_SUFFIX[weights]).csv"

    search_dirs = [
        ESTIMATES_DIR,
    ]
    csv_path = ""
    for d in search_dirs
        candidate = joinpath(d, fname)
        isfile(candidate) && (csv_path = candidate; break)
    end
    isempty(csv_path) &&
        error("CSV '$fname' not found. Searched:\n  " * join(search_dirs, "\n  "))

    params   = copy(DEFAULT_PARAMS)
    n_loaded = 0

    open(csv_path) do io
        for raw in eachline(io)
            line = strip(raw)
            isempty(line)                  && continue
            startswith(line, "block")      && continue
            startswith(line, "---")        && continue
            startswith(line, "Q ")         && continue
            startswith(line, "converged")  && continue
            startswith(line, "iterations") && continue

            local block::String, name::String, estimate::Float64, is_fixed::Bool

            if occursin(',', line)
                parts = split(line, ',')
                length(parts) < 7 && continue
                block    = strip(parts[1])
                name     = strip(parts[2])
                is_fixed = strip(parts[7]) == "true"
                est      = tryparse(Float64, strip(parts[4]))
            else
                tokens = split(line)
                length(tokens) < 6 && continue
                block    = tokens[1]
                name     = tokens[2]
                is_fixed = tokens[end] == "true"
                est      = tryparse(Float64, tokens[end-3])
            end

            isnothing(est) && continue
            # NOTE: do NOT skip fixed rows — crisis estimation pins deep structural
            # params in fixed rows; we need those baseline values in the notebook.

            key = get(CSV_KEY_MAP, (block, name), nothing)
            isnothing(key) && continue

            params[key] = est
            n_loaded   += 1
        end
    end

    println("  ✓ Loaded $n_loaded params from: $csv_path")
    return params
end

# ── Load ──────────────────────────────────────────────────────────────────────
p = USE_DEFAULT_PARAMS ? copy(DEFAULT_PARAMS) :
                         load_scenario_params(ACTIVE_SCENARIO, ACTIVE_WEIGHTS)

# ── Build structs (adjust field names to match your params.jl) ────────────────
common = CommonParams(
    r   = p[:r],
    ν   = p[:nu],
    φ   = p[:phi],
    a_ℓ = p[:a_l],
    b_ℓ = p[:b_l],
    c   = p[:c],
)

regime = RegimeParams(
    PU    = p[:PU],
    gamma_PS = p[:gamma_PS],
    bU    = p[:bU],
    bT    = p[:bT],
    bS    = p[:bS],
    α_U   = p[:alpha_U],
    a_Γ   = p[:a_Gam],
    b_Γ   = p[:b_Gam],
)

unsk_par = UnskilledParams(
    μ = p[:unsk_mu],
    η = p[:unsk_eta],
    k = p[:unsk_k],
    β = p[:unsk_bet],
    λ = p[:unsk_lam],
)

skl_par = SkilledParams(
    μ = p[:skl_mu],
    η = p[:skl_eta],
    k = p[:skl_k],
    β = p[:skl_bet],
    ξ = p[:skl_xi],
    λ = p[:skl_lam],
    σ = p[:skl_sig],
)


println("Active scenario : ", ACTIVE_SCENARIO, "  |  weights : ", ACTIVE_WEIGHTS)
println("Parameters defined: ", length(PARAM_KEYS), " free params")

# ── Solve the stationary equilibrium ────────────────────────────────────────
println("Solving model...")
@time model, result = solve_model(common, regime, unsk_par, skl_par, sim;
                                   Nx=200, Np_U=200, Np_S=200)

if result.ok
    @printf("Solver converged  (U=%s  S=%s  global=%s)\n",
            result.converged_U, result.converged_S, result.converged_global)
else
    @printf("WARNING: solver did not fully converge  (U=%s  S=%s  global=%s)\n",
            result.converged_U, result.converged_S, result.converged_global)
end

# ── Equilibrium objects and population accounting ────────────────────────────
obj = compute_equilibrium_objects(model)
print_accounting(obj)

# ── 5b  Moment diagnostics ──────────────────────────────────────────────────
# Load the moments helper from the sibling smm/ directory.
# moments.jl requires CSV + DataFrames for load_data_moments.


# ── Compute model moments from the solved equilibrium ───────────────────────
mm = model_moments(obj)

# ── Try to load empirical targets for side-by-side comparison ───────────────
# Adjust DERIVED_DIR to wherever your data pipeline writes derived/ CSVs.

_dm, _data_ok = nothing, false
try
    _dm     = load_data_moments(window = Symbol(ACTIVE_SCENARIO),
                                derived_dir = DERIVED_DIR)
    _data_ok = true
catch e
    @warn "Data moments not loaded — showing model values only.\n  ($e)"
end

# ── Print table ──────────────────────────────────────────────────────────────
sep_wide  = "="^70
sep_thin  = "-"^70
println("\n", sep_wide)
println("  MOMENT DIAGNOSTICS")
println("  scenario : $ACTIVE_SCENARIO    weights : $ACTIVE_WEIGHTS")
println(sep_wide)

if _data_ok
    @printf("  %-22s  %12s  %12s  %10s\n",
            "Moment", "Model", "Data target", "Δ (m−d)")
    println(sep_thin)
    for name in MOMENT_NAMES
        mv = getfield(mm,  name)
        dv = getfield(_dm, name).value
        flag = abs(mv - dv) > 0.5 * abs(dv) + 1e-8 ? "  ◄" : ""   # flag large gaps
        @printf("  %-22s  %12.6f  %12.6f  %+10.4f%s\n",
                string(name), mv, dv, mv - dv, flag)
    end
else
    @printf("  %-22s  %14s\n", "Moment", "Model value")
    println(sep_thin)
    for name in MOMENT_NAMES
        @printf("  %-22s  %14.6f\n", string(name), getfield(mm, name))
    end
end

println(sep_wide)

# ── Compute Q from the current model moments and the active SMM spec ─────────
Q_val = NaN
if _data_ok
    # 1. Build the weight matrix consistent with ACTIVE_WEIGHTS
    _cond_target = ACTIVE_WEIGHTS == :full       ? 1e8 :
                   ACTIVE_WEIGHTS == :diagonal   ? 0.0 :
                   ACTIVE_WEIGHTS == :compressed ? 1.0 :
                   2.0   # :equal → nothing

    _W = try
        load_weight_matrix(window        = Symbol(ACTIVE_SCENARIO),
                           derived_dir   = DERIVED_DIR,
                           cond_target   = _cond_target,
                           skip_moments  = SKIP_MOMENTS)
    catch
        nothing   # fall back to diagonal / equal weights if files are absent
    end

    # 2. Build a minimal SMMSpec (no free params needed — we just need
    #    spec.moments and spec.W so compute_loss / compute_loss_matrix work).
    #    Use the same SKIP_MOMENTS as smm_main.jl so W size matches the loss function.
    _diag_run  = SMMRunParams(w_cond_target = _cond_target)
    _spec_diag = build_smm_spec(_dm, sim;
                                free_specs   = ParamSpec[],
                                run          = _diag_run,
                                W            = _W,
                                skip_moments = SKIP_MOMENTS)

    # 3. Evaluate Q — spec already holds _W, so just pass spec
    Q_val = if !isnothing(_W)
        compute_loss_matrix(mm, _spec_diag, _spec_diag.W)
    else
        compute_loss(mm, _spec_diag)
    end
else
    @warn "Q cannot be computed — data moments not loaded."
end


println("\n", sep_wide)
println("  MOMENT DIAGNOSTICS")
println("  scenario : $ACTIVE_SCENARIO    weights : $ACTIVE_WEIGHTS")
@printf("  Q(spec)  : %.8e\n", Q_val)
println(sep_wide)

# ── Corner-solution alert ────────────────────────────────────────────────────
corner_flags = String[]
mm.ur_U        ≈ 1.0            && push!(corner_flags, "ur_U = 1  (all unskilled unemployed)")
mm.skilled_share < 1e-6         && push!(corner_flags, "skilled_share ≈ 0  (empty skilled sector)")
mm.training_share < 1e-6        && push!(corner_flags, "training_share ≈ 0  (no one training)")
mm.mean_wage_U   < 1e-6         && push!(corner_flags, "mean_wage_U ≈ 0")
mm.mean_wage_S   < 1e-6         && push!(corner_flags, "mean_wage_S ≈ 0")

if !isempty(corner_flags)
    println("\n  ⚠  CORNER SOLUTION DETECTED:")
    for f in corner_flags
        println("     • ", f)
    end
    println()
end

ErrorException: CSV 'smm_estimates_crisis_fc_equalW.csv' not found. Searched:
  /Users/ramzi.chariag/Documents/CEU/PhD/Research/Search and matching/code/notebooks/../../output/tables

## 3.1 · SMM Diagnostics

## 6 · Stationary densities

In [4]:
# ── Plot output directory ─────────────────────────────────────────────────
# Use `global` so the cell is re-runnable (const would error on second run)
global SINGLE_RUN_DIR = joinpath(OUTPUT_DIR, "plots", "single_run")
mkpath(SINGLE_RUN_DIR)   # creates the full path if it doesn't exist

# Build a suffix string: "_" + scenario tag + weight suffix (W_SUFFIX already starts with "_")
global _PLOT_SUFFIX = "_" * SCENARIO_TAGS[ACTIVE_SCENARIO] * W_SUFFIX[ACTIVE_WEIGHTS]

"""Save a figure to SINGLE_RUN_DIR with the given base name + scenario/weight suffix."""
function _save_plot(fig, basename::String; ext=".png")
    path = joinpath(SINGLE_RUN_DIR, basename * _PLOT_SUFFIX * ext)
    savefig(fig, path)
    println("  ✓ saved: ", path)
    # Reset GR backend state after every figure so that viewport/window
    # settings from one plot (e.g. xlims outside [0,1]) never bleed into
    # the next one.
    gr()
    default(fmt = :png)
end

# Force PNG-only display in the notebook.
# The GR/SVG backend uses a global clip-path ID counter that increments with
# every plot.  When a cell is re-run the new SVGs carry higher IDs, but the
# old SVGs from other cells are still in the notebook DOM — shared IDs collide
# and plots render incorrectly (garbled / blank / wrong clipping).
# PNG avoids this entirely because it is a raster image with no DOM references.
ENV["GKSwstype"] = "100"          # GR: write to file only (no window)
default(fmt = :png)                # Plots.jl: only emit PNG to the notebook

_set_theme!()

let fig = fig_densities(obj)
    display(fig)
    _save_plot(fig, "densities")
end


UndefVarError: UndefVarError: `obj` not defined

## 7 · Unskilled unemployment values and training policy

In [5]:
let fig = fig_unskilled_values(obj)
    display(fig)
    _save_plot(fig, "unskilled_values")
end

let fig = fig_training_policy(obj)
    display(fig)
    _save_plot(fig, "training_policy")
end

UndefVarError: UndefVarError: `obj` not defined

## 8 · Employment heatmaps

In [6]:
let fig = fig_employment_heatmaps(obj)
    display(fig)
    _save_plot(fig, "employment_heatmaps")
end

let fig = fig_total_employment(obj)
    display(fig)
    _save_plot(fig, "total_employment")
end


UndefVarError: UndefVarError: `obj` not defined

### 8.1 · Skilled employment by P_S(x)

In [7]:
# Skilled employment density in (P_S(x), p) space.
# P_S(x) = γ · x^{γ−1} is the skilled productivity index; plotting against it
# stretches the type axis nonlinearly so that high-productivity workers
# (large P_S) are given proportionally more space than in the raw (x, p) view.
# Pass regime.gamma_PS so the function knows the shape parameter γ.
let fig = fig_skilled_employment_by_PS(obj, regime.gamma_PS)
    display(fig)
    _save_plot(fig, "skilled_employment_by_PS")
end

UndefVarError: UndefVarError: `obj` not defined

## 9 · Unskilled frontier firm value and reservation rule

In [8]:
let fig = fig_unskilled_frontier(obj)
    display(fig)
    _save_plot(fig, "unskilled_frontier")
end


UndefVarError: UndefVarError: `obj` not defined

## 10 · Unskilled value surfaces

In [9]:
let fig = fig_unskilled_value_surfaces(obj)
    display(fig)
    _save_plot(fig, "unskilled_value_surfaces")
end


UndefVarError: UndefVarError: `obj` not defined

## 11 · Skilled cutoffs and value surfaces

In [10]:
let fig = fig_skilled_cutoffs(obj)
    display(fig)
    _save_plot(fig, "skilled_cutoffs")
end

let fig = fig_skilled_worker_values(obj)
    display(fig)
    _save_plot(fig, "skilled_worker_values")
end

let fig = fig_skilled_firm_values(obj)
    display(fig)
    _save_plot(fig, "skilled_firm_values")
end


UndefVarError: UndefVarError: `obj` not defined

## 12 · Surplus heatmaps and unemployment values

In [11]:
let fig = fig_surplus_heatmaps(obj)
    display(fig)
    _save_plot(fig, "surplus_heatmaps")
end

let fig = fig_unemployment_values(obj)
    display(fig)
    _save_plot(fig, "unemployment_values")
end


UndefVarError: UndefVarError: `obj` not defined

## 13 · Wages

In [12]:
let fig = fig_unskilled_wage(obj)
    display(fig)
    _save_plot(fig, "unskilled_wage")
end

let fig = fig_skilled_wages(obj)
    display(fig)
    _save_plot(fig, "skilled_wages")
end

let fig = fig_skill_premium(obj)
    display(fig)
    _save_plot(fig, "skill_premium")
end

let fig = fig_wage_densities(obj)
    display(fig)
    _save_plot(fig, "wage_densities")
end

let fig = fig_wage_densities_pooled(obj)
    display(fig)
    _save_plot(fig, "wage_densities_pooled")
end

let fig = fig_wage_pooled_density(obj)
    display(fig)
    _save_plot(fig, "wage_pooled_density")
end


UndefVarError: UndefVarError: `obj` not defined

## 14 · 3-D skilled surplus surface

In [13]:
let fig = surface(
    obj.pg, obj.xg, obj.Smax_surface,
    xlabel = L"p", ylabel = L"x",
    zlabel = L"\max(S^0, S^1)",
    title  = "Skilled surplus surface",
    color  = :viridis,
    camera = (30, 35),
    size   = (700, 500),
)
    display(fig)
    _save_plot(fig, "skilled_surplus_surface")
end


UndefVarError: UndefVarError: `obj` not defined

In [14]:
# employment surface
let fig = surface(
    obj.pg, obj.xg, obj.eU_surface,
    xlabel = L"p", ylabel = L"x",
    zlabel = "employment",
    title  = "Unskilled Employment surface",
    color  = :viridis,
    camera = (30, 35),
    size   = (700, 500),
)
    display(fig)
    _save_plot(fig, "employment_surface")
end


UndefVarError: UndefVarError: `obj` not defined